# Matmul - Tensor Core Triton Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qooba/bielik-anatomy-triton/blob/main/notebooks/ep02-matmul.ipynb)

Part of the [Bielik Anatomy](https://github.com/qooba/bielik-anatomy-triton) series - Episode 2: Matmul. Full write-up: [ep02-matmul.md](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep02-matmul.md).

Matrix multiplication is the single most frequent operation in a transformer - roughly 80% of Bielik's forward-pass compute. This notebook runs the Tensor Core optimized Triton matmul kernel (`kernels/matmul/matmul.py`) and compares it against PyTorch/cuBLAS and a naive Triton baseline.

This notebook lets you run this kernel's correctness check and benchmark on a free Colab GPU - no local NVIDIA hardware required.

**Runtime:** select a GPU runtime before running: `Runtime -> Change runtime type -> T4 GPU` (free tier works fine).

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_DIR = "bielik-anatomy-triton"
if not os.path.isdir(REPO_DIR):
    !git clone --quiet https://github.com/qooba/bielik-anatomy-triton {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 1. Correctness check

Compare the Triton kernel output against a plain PyTorch reference implementation.

In [ ]:
import torch
import triton
from kernels.matmul.matmul import matmul as matmul_tensorcore_kernel

torch.manual_seed(0)
M, K, N = 512, 512, 512
a = torch.randn((M, K), device="cuda", dtype=torch.bfloat16)
b = torch.randn((K, N), device="cuda", dtype=torch.bfloat16)
c = torch.empty((M, N), device="cuda", dtype=torch.bfloat16)

def grid(META):
    return (triton.cdiv(M, META["BLOCK_SIZE_M"]) * triton.cdiv(N, META["BLOCK_SIZE_N"]),)

matmul_tensorcore_kernel[grid](
    a, b, c, M, N, K,
    a.stride(0), a.stride(1),
    b.stride(0), b.stride(1),
    c.stride(0), c.stride(1),
)

ref = torch.matmul(a, b)
max_diff = (c - ref).abs().max().item()
print(f"max abs diff vs cuBLAS: {max_diff:.4f}")
assert torch.allclose(c, ref, atol=1e-1, rtol=5e-2)
print("Correctness check passed.")

## 2. Benchmark

Runs the same sweep as `make benchmark-matmul` and regenerates the plots used in the episode doc. Takes roughly 1-3 minutes on a T4.

> **Note on T4 (Colab free tier):** T4 is Turing (compute capability 7.5), which has no native BF16 tensor cores - those arrived with Ampere (RTX 30-series, A100, and up). This kernel benchmarks BF16 `tl.dot` matmuls, so on T4 both the Triton and PyTorch BF16 providers fall back to a slow, non-tensor-core path. The TFLOPS numbers below will look much worse than the docs (measured on an RTX 4060 Ti, Ada) and are **not representative of the kernel's real performance** - for numbers that match the docs, switch to an A100/L4 runtime (Colab Pro) via `Runtime -> Change runtime type`.

In [ ]:
!PYTHONPATH=. python benchmarks/matmul/benchmark_matmul_guide.py --save-plots --plot-dir=docs/plots/matmul/

## 3. Results

In [ ]:
from IPython.display import Image, display

plot_dir = "docs/plots/matmul/"
for fname in [
    "matmul-tflops-vs-size.png",
    "matmul-tflops-vs-shape-bielik.png",
    "matmul-summary-bielik-shapes-latency_(ms).png",
    "matmul-summary-bielik-shapes-tflops.png",
]:
    display(Image(filename=f"{plot_dir}{fname}"))

---
[Full episode write-up](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep02-matmul.md) · [Series overview](https://github.com/qooba/bielik-anatomy-triton/blob/main/README.md)